# 🔧 Bosch Accelerometer Motion Event Explainer

**Goal:** Build an end-to-end pipeline that:
1. Generates synthetic Bosch-like accelerometer data
2. Normalizes and engineers features
3. Detects motion events using rule-based logic
4. Summarizes events in plain language
5. Indexes summaries with LlamaIndex for natural-language queries

**Schema:** `timestamp, accel_x, accel_y, accel_z`

**No external dataset required** — everything is generated synthetically inside this notebook.

---
## 1. Setup

Install all required libraries. LlamaIndex is used for indexing and querying event summaries.
If no OpenAI API key is found, a template-based fallback will be used automatically.

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib llama-index --quiet

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Optional

warnings.filterwarnings('ignore')

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('✅ Libraries loaded successfully.')

---
## 2. Synthetic Data Generation

We generate four motion segments that mimic real Bosch accelerometer behavior:

| Segment | Description |
|---|---|
| **idle** | Device at rest, near-zero acceleration |
| **walking_like** | Rhythmic, low-frequency oscillation |
| **impact** | Short, high-magnitude spike |
| **shaking** | High-frequency, high-amplitude noise |

All segments are combined into a single DataFrame with columns: `timestamp, accel_x, accel_y, accel_z`.

In [ ]:
# ── Tunable sampling parameters ──────────────────────────────────────────────
SAMPLE_RATE_HZ = 100          # Samples per second
GRAVITY = 9.81                # m/s² — baseline gravity offset on Z axis

# ── Segment durations (seconds) ──────────────────────────────────────────────
SEG_DURATIONS = {
    'idle':         5.0,
    'walking_like': 8.0,
    'impact':       2.0,
    'shaking':      4.0,
}


def make_idle(n: int) -> np.ndarray:
    """Device at rest — near-zero XY, gravity on Z, tiny sensor noise."""
    noise = 0.05
    x = np.random.normal(0, noise, n)
    y = np.random.normal(0, noise, n)
    z = np.random.normal(GRAVITY, noise, n)
    return np.column_stack([x, y, z])


def make_walking_like(n: int) -> np.ndarray:
    """Rhythmic gait-like oscillation at ~2 Hz with moderate noise."""
    t = np.linspace(0, n / SAMPLE_RATE_HZ, n)
    freq = 2.0        # Hz — step frequency
    amp  = 1.5        # m/s² — stride amplitude
    noise = 0.2
    x = amp * np.sin(2 * np.pi * freq * t) + np.random.normal(0, noise, n)
    y = amp * np.cos(2 * np.pi * freq * t) + np.random.normal(0, noise, n)
    z = GRAVITY + 0.5 * np.sin(2 * np.pi * freq * t) + np.random.normal(0, noise, n)
    return np.column_stack([x, y, z])


def make_impact(n: int) -> np.ndarray:
    """Single sharp spike followed by damped ringing, then quiet."""
    x = np.random.normal(0, 0.1, n)
    y = np.random.normal(0, 0.1, n)
    z = np.random.normal(GRAVITY, 0.1, n)
    # Place the spike at 20 % into the segment
    spike_idx = max(1, int(0.20 * n))
    spike_width = max(2, int(0.03 * n))
    envelope = np.zeros(n)
    for i in range(spike_width):
        idx = spike_idx + i
        if idx < n:
            envelope[idx] = 30.0 * np.exp(-4.0 * i / spike_width)  # decaying spike
    z += envelope
    x += envelope * 0.4
    return np.column_stack([x, y, z])


def make_shaking(n: int) -> np.ndarray:
    """High-frequency, high-amplitude random vibration."""
    amp = 5.0
    x = np.random.uniform(-amp, amp, n)
    y = np.random.uniform(-amp, amp, n)
    z = GRAVITY + np.random.uniform(-amp, amp, n)
    return np.column_stack([x, y, z])


def generate_synthetic_data(
    sample_rate: int = SAMPLE_RATE_HZ,
    durations: dict = SEG_DURATIONS,
    seed: int = RANDOM_SEED
) -> pd.DataFrame:
    """
    Build all segments in order and return a single DataFrame.
    Columns: timestamp (s), accel_x, accel_y, accel_z
    """
    np.random.seed(seed)
    generators = {
        'idle':         make_idle,
        'walking_like': make_walking_like,
        'impact':       make_impact,
        'shaking':      make_shaking,
    }
    segments = []
    t_offset = 0.0
    for seg_name, duration in durations.items():
        n = int(duration * sample_rate)
        data = generators[seg_name](n)
        ts = t_offset + np.arange(n) / sample_rate
        df_seg = pd.DataFrame(data, columns=['accel_x', 'accel_y', 'accel_z'])
        df_seg.insert(0, 'timestamp', np.round(ts, 4))
        df_seg['_segment_label'] = seg_name   # kept for visual reference
        segments.append(df_seg)
        t_offset = ts[-1] + 1 / sample_rate

    df = pd.concat(segments, ignore_index=True)
    return df


# Generate the data
raw_df = generate_synthetic_data()

print(f'Generated {len(raw_df):,} samples  |  Duration: {raw_df["timestamp"].max():.2f}s')
print(f'Columns: {list(raw_df.columns)}')
raw_df.head()

---
## 3. Optional CSV Upload

**Skip this cell if you want to use the synthetic data.** 
Upload a real CSV with columns `timestamp, accel_x, accel_y, accel_z` to override the synthetic dataset.

> The upload widget only works inside Google Colab.

In [ ]:
# ── Optional CSV upload (Colab only) ─────────────────────────────────────────
REQUIRED_COLS = {'timestamp', 'accel_x', 'accel_y', 'accel_z'}

def try_upload_csv(current_df: pd.DataFrame) -> pd.DataFrame:
    """Attempt to upload a CSV in Colab; fall back to current_df if unavailable."""
    try:
        from google.colab import files
        print('Colab detected — launching file picker...')
        uploaded = files.upload()
        if not uploaded:
            print('No file uploaded. Using synthetic data.')
            return current_df
        fname = list(uploaded.keys())[0]
        import io
        df_up = pd.read_csv(io.BytesIO(uploaded[fname]))
        missing = REQUIRED_COLS - set(df_up.columns)
        if missing:
            print(f'⚠️  Missing columns: {missing}. Using synthetic data.')
            return current_df
        df_up = df_up[list(REQUIRED_COLS)].copy()
        df_up['_segment_label'] = 'uploaded'
        print(f'✅ Loaded {len(df_up):,} rows from "{fname}"')
        return df_up
    except ImportError:
        print('Not running in Colab — skipping upload. Using synthetic data.')
        return current_df


# Comment out the next line to always use synthetic data
# raw_df = try_upload_csv(raw_df)

print('Using current dataset:', len(raw_df), 'rows')

---
## 4. Data Normalization

We standardize columns and apply **z-score normalization** to the three axes so that feature thresholds remain scale-independent. The raw values are preserved in `_raw_*` columns.

In [ ]:
ACCEL_COLS = ['accel_x', 'accel_y', 'accel_z']


def normalize_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate required columns, drop NaNs, and z-score normalize XYZ axes.
    Raw values are preserved as _raw_* columns.
    """
    df = df.copy()

    # Validate columns
    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    # Drop rows with NaN in critical columns
    before = len(df)
    df.dropna(subset=['timestamp'] + ACCEL_COLS, inplace=True)
    dropped = before - len(df)
    if dropped:
        print(f'Dropped {dropped} NaN rows.')

    # Sort by timestamp
    df.sort_values('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Z-score normalization — preserve raw values
    for col in ACCEL_COLS:
        df[f'_raw_{col}'] = df[col]
        mu, sigma = df[col].mean(), df[col].std()
        df[col] = (df[col] - mu) / (sigma if sigma > 0 else 1.0)

    print(f'✅ Normalized {len(df):,} rows. Axes z-scored.')
    return df


norm_df = normalize_data(raw_df)
norm_df[['timestamp'] + ACCEL_COLS].describe().round(3)

---
## 5. Feature Engineering

Derived features used for event detection:

| Feature | Formula / Method |
|---|---|
| `accel_magnitude` | √(x² + y² + z²) |
| `rolling_mean` | 50-sample rolling mean of magnitude |
| `rolling_std` | 50-sample rolling std of magnitude |

In [ ]:
# ── Tunable window ────────────────────────────────────────────────────────────
ROLLING_WINDOW = 50   # samples (~0.5 s at 100 Hz)


def engineer_features(df: pd.DataFrame, window: int = ROLLING_WINDOW) -> pd.DataFrame:
    """Add magnitude and rolling statistics."""
    df = df.copy()

    # Euclidean magnitude across all three axes
    df['accel_magnitude'] = np.sqrt(
        df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2
    )

    # Rolling statistics (min_periods=1 avoids NaN at edges)
    df['rolling_mean'] = (
        df['accel_magnitude'].rolling(window, min_periods=1).mean()
    )
    df['rolling_std'] = (
        df['accel_magnitude'].rolling(window, min_periods=1).std().fillna(0)
    )

    print(f'✅ Features added. Magnitude range: '
          f'[{df["accel_magnitude"].min():.3f}, {df["accel_magnitude"].max():.3f}]')
    return df


feat_df = engineer_features(norm_df)
feat_df[['timestamp', 'accel_magnitude', 'rolling_mean', 'rolling_std']].head(10)

---
## 6. Rule-Based Event Detection

Events are detected by scanning the rolling statistics against tunable thresholds:

| Event | Rule |
|---|---|
| **idle** | rolling_std < `IDLE_STD_MAX` |
| **impact** | rolling_mean > `IMPACT_MAG_MIN` AND rolling_std > `IMPACT_STD_MIN` |
| **shaking** | rolling_std > `SHAKE_STD_MIN` (but not impact) |
| **walking_like** | rolling_mean in (`WALK_MAG_MIN`, `WALK_MAG_MAX`) AND rolling_std in (`WALK_STD_MIN`, `WALK_STD_MAX`) |

Consecutive same-label samples are merged into a single event object.

In [ ]:
# ── Tunable thresholds ────────────────────────────────────────────────────────
IDLE_STD_MAX     = 0.15   # rolling_std below this → idle
IMPACT_MAG_MIN   = 2.5    # rolling_mean above this → candidate impact
IMPACT_STD_MIN   = 1.0    # rolling_std above this → confirmed impact
SHAKE_STD_MIN    = 1.8    # rolling_std above this → shaking
WALK_MAG_MIN     = 0.8    # rolling_mean lower bound for walking
WALK_MAG_MAX     = 2.5    # rolling_mean upper bound for walking
WALK_STD_MIN     = 0.10   # rolling_std lower bound for walking
WALK_STD_MAX     = 1.8    # rolling_std upper bound for walking

# Minimum event duration in samples to suppress noise
MIN_EVENT_SAMPLES = 10


@dataclass
class MotionEvent:
    start_time:       float
    end_time:         float
    event_type:       str
    max_magnitude:    float
    mean_magnitude:   float
    explanation_seed: str  # short phrase used to build the summary


def classify_sample(row: pd.Series) -> str:
    """Return event label for a single row based on thresholds."""
    rmean = row['rolling_mean']
    rstd  = row['rolling_std']

    if rmean > IMPACT_MAG_MIN and rstd > IMPACT_STD_MIN:
        return 'impact'
    if rstd > SHAKE_STD_MIN:
        return 'shaking'
    if (WALK_MAG_MIN <= rmean <= WALK_MAG_MAX and
            WALK_STD_MIN <= rstd <= WALK_STD_MAX):
        return 'walking_like'
    if rstd < IDLE_STD_MAX:
        return 'idle'
    return 'unknown'


def detect_events(df: pd.DataFrame) -> List[MotionEvent]:
    """
    Classify every sample, then merge consecutive same-label runs
    into MotionEvent objects.
    """
    df = df.copy()
    df['event_label'] = df.apply(classify_sample, axis=1)

    events: List[MotionEvent] = []
    # Group consecutive rows with the same label
    df['_run_id'] = (df['event_label'] != df['event_label'].shift()).cumsum()

    for _, grp in df.groupby('_run_id'):
        if len(grp) < MIN_EVENT_SAMPLES:
            continue   # skip very short transients
        etype = grp['event_label'].iloc[0]
        mag   = grp['accel_magnitude']

        # Build a short seed phrase describing the event
        seed_map = {
            'idle':         'the device was stationary with minimal movement',
            'walking_like': 'rhythmic oscillation consistent with walking or repetitive motion',
            'impact':       'a sudden high-magnitude spike indicating a physical impact or drop',
            'shaking':      'high-frequency high-amplitude vibration indicating vigorous shaking',
            'unknown':      'unclassified motion not matching any known pattern',
        }

        events.append(MotionEvent(
            start_time       = grp['timestamp'].iloc[0],
            end_time         = grp['timestamp'].iloc[-1],
            event_type       = etype,
            max_magnitude    = round(float(mag.max()), 4),
            mean_magnitude   = round(float(mag.mean()), 4),
            explanation_seed = seed_map.get(etype, ''),
        ))

    return events, df  # also return df with labels attached


events, labeled_df = detect_events(feat_df)

print(f'✅ Detected {len(events)} events.')
pd.DataFrame([
    {
        'event_type':     e.event_type,
        'start_time':     e.start_time,
        'end_time':       e.end_time,
        'duration_s':     round(e.end_time - e.start_time, 3),
        'max_magnitude':  e.max_magnitude,
        'mean_magnitude': e.mean_magnitude,
    }
    for e in events
])

---
## 7. Event Summarization

Each `MotionEvent` is converted into a concise, human-readable English sentence. These summaries become the documents indexed by LlamaIndex.

In [ ]:
def summarize_event(event: MotionEvent) -> str:
    """
    Produce a readable one-sentence summary of a MotionEvent.
    Uses the explanation_seed as the descriptive phrase.
    """
    duration = round(event.end_time - event.start_time, 2)
    summary = (
        f"[{event.event_type.upper()}] From t={event.start_time:.2f}s to "
        f"t={event.end_time:.2f}s ({duration}s): {event.explanation_seed}. "
        f"Peak magnitude: {event.max_magnitude:.3f}, "
        f"mean magnitude: {event.mean_magnitude:.3f}."
    )
    return summary


def build_summaries(events: List[MotionEvent]) -> List[str]:
    """Return a list of summary strings, one per event."""
    return [summarize_event(e) for e in events]


summaries = build_summaries(events)

print(f'Generated {len(summaries)} summaries:\n')
for s in summaries:
    print(' •', s)

---
## 7b. Visualization

Three matplotlib plots:
1. Raw accelerometer axes over time
2. Acceleration magnitude with detected event bands highlighted
3. Rolling statistics (mean and std)

In [ ]:
# Color map for event types
EVENT_COLORS = {
    'idle':         '#6EC6E6',
    'walking_like': '#82C882',
    'impact':       '#E85858',
    'shaking':      '#F5A623',
    'unknown':      '#CCCCCC',
}

ts = labeled_df['timestamp'].values

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Bosch Accelerometer Motion Event Explainer', fontsize=14, fontweight='bold')

# ── Plot 1: Raw axes ──────────────────────────────────────────────────────────
ax1 = axes[0]
for col, color, label in zip(
    ACCEL_COLS,
    ['#4A90D9', '#E87040', '#5CB85C'],
    ['X', 'Y', 'Z']
):
    ax1.plot(ts, labeled_df[col], lw=0.7, alpha=0.8, color=color, label=f'Accel {label}')
ax1.set_ylabel('Normalized Accel')
ax1.set_title('Accelerometer Axes (z-scored)')
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Magnitude + event bands ──────────────────────────────────────────
ax2 = axes[1]
ax2.plot(ts, labeled_df['accel_magnitude'], lw=0.8, color='#333333', label='Magnitude', zorder=3)
for evt in events:
    ax2.axvspan(
        evt.start_time, evt.end_time,
        alpha=0.25,
        color=EVENT_COLORS.get(evt.event_type, '#CCCCCC'),
        label=evt.event_type
    )
ax2.set_ylabel('Magnitude')
ax2.set_title('Acceleration Magnitude + Detected Events')
ax2.grid(True, alpha=0.3)
# Deduplicate legend entries
handles, labels_ = ax2.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels_):
    seen.setdefault(l, h)
ax2.legend(seen.values(), seen.keys(), loc='upper right', fontsize=8)

# ── Plot 3: Rolling statistics ────────────────────────────────────────────────
ax3 = axes[2]
ax3.plot(ts, labeled_df['rolling_mean'], lw=1.0, color='#4A90D9', label='Rolling Mean')
ax3.plot(ts, labeled_df['rolling_std'],  lw=1.0, color='#E87040', label='Rolling Std', linestyle='--')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Value')
ax3.set_title(f'Rolling Statistics (window = {ROLLING_WINDOW} samples)')
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accelerometer_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as accelerometer_overview.png')

---
## 8. LlamaIndex Integration

Event summaries are indexed using LlamaIndex's `VectorStoreIndex`.

**API key handling:**
- If `OPENAI_API_KEY` is set, LlamaIndex will use GPT embeddings and completion.
- If no key is found, a lightweight **template-based fallback** answers queries by keyword matching against the summaries.

> Set your key with: `import os; os.environ['OPENAI_API_KEY'] = 'sk-...'`

In [ ]:
HAS_API_KEY = bool(os.environ.get('OPENAI_API_KEY', '').strip())

# ── LlamaIndex index builder ──────────────────────────────────────────────────
query_engine = None   # will be set below if API key available

if HAS_API_KEY:
    try:
        from llama_index.core import VectorStoreIndex, Document
        documents = [Document(text=s) for s in summaries]
        index = VectorStoreIndex.from_documents(documents)
        query_engine = index.as_query_engine()
        print('✅ LlamaIndex VectorStoreIndex built with OpenAI embeddings.')
    except Exception as e:
        print(f'⚠️  LlamaIndex init failed ({e}). Falling back to template answers.')
        HAS_API_KEY = False
else:
    print('ℹ️  No OPENAI_API_KEY found. Using template-based fallback query engine.')


# ── Template-based fallback ───────────────────────────────────────────────────
def keyword_query(question: str, summaries: List[str]) -> str:
    """
    Lightweight keyword-matching fallback when no LLM is available.
    Returns matching summaries or a 'not found' message.
    """
    q = question.lower()

    # Keyword → event type mapping
    keyword_map = {
        'abnormal':  ['impact', 'shaking', 'unknown'],
        'unusual':   ['impact', 'shaking', 'unknown'],
        'impact':    ['impact'],
        'drop':      ['impact'],
        'collision': ['impact'],
        'idle':      ['idle'],
        'stationary':['idle'],
        'still':     ['idle'],
        'rest':      ['idle'],
        'walk':      ['walking_like'],
        'walking':   ['walking_like'],
        'shake':     ['shaking'],
        'shaking':   ['shaking'],
        'vibrat':    ['shaking'],
    }

    matched_types = set()
    for kw, types in keyword_map.items():
        if kw in q:
            matched_types.update(types)

    if not matched_types:
        # Return all summaries if no keyword match
        matches = summaries
        header  = 'Here are all detected events:'
    else:
        matches = [
            s for s in summaries
            if any(t.upper() in s for t in matched_types)
        ]
        header = f'Events matching your query ({", ".join(matched_types)}):'

    if not matches:
        return f'No events found matching: "{question}"'

    return header + '\n' + '\n'.join(f'  • {m}' for m in matches)


def query_events(question: str) -> str:
    """Unified query interface — uses LlamaIndex if available, else fallback."""
    if HAS_API_KEY and query_engine is not None:
        response = query_engine.query(question)
        return str(response)
    else:
        return keyword_query(question, summaries)


print('\nQuery engine ready.')

---
## 9. Query Examples

Run natural-language queries against the indexed event summaries.

In [ ]:
EXAMPLE_QUERIES = [
    'What abnormal events happened?',
    'When was the device idle?',
    'Did any impact occur?',
    'Was there any walking detected?',
    'Describe all shaking events.',
]

for q in EXAMPLE_QUERIES:
    print(f'\n❓ Query: "{q}"')
    print('─' * 60)
    answer = query_events(q)
    print(answer)

---
## 10. Save Outputs

Save the following files:
- `synthetic_accel_data.csv` — raw synthetic accelerometer readings
- `detected_events.csv` — all detected motion events with metadata
- `accelerometer_overview.png` — already saved in Section 7b

In [ ]:
# ── Save synthetic data (original schema only) ────────────────────────────────
output_cols = ['timestamp', 'accel_x', 'accel_y', 'accel_z']
# Use raw (un-normalized) values for the saved CSV
raw_export = raw_df[output_cols + ['_segment_label']].copy()
raw_export.to_csv('synthetic_accel_data.csv', index=False)
print('✅ Saved synthetic_accel_data.csv')

# ── Save detected events ──────────────────────────────────────────────────────
events_rows = [
    {
        'event_type':       e.event_type,
        'start_time':       e.start_time,
        'end_time':         e.end_time,
        'duration_s':       round(e.end_time - e.start_time, 3),
        'max_magnitude':    e.max_magnitude,
        'mean_magnitude':   e.mean_magnitude,
        'explanation_seed': e.explanation_seed,
        'summary':          summarize_event(e),
    }
    for e in events
]
events_df = pd.DataFrame(events_rows)
events_df.to_csv('detected_events.csv', index=False)
print('✅ Saved detected_events.csv')

print('\n--- Detected Events Table ---')
events_df[['event_type', 'start_time', 'end_time', 'duration_s',
           'max_magnitude', 'mean_magnitude']]

---
## 11. Next Steps for V2

Ideas to extend this notebook into a production-grade system:

**Signal Processing**
- Add a Butterworth low-pass filter to remove sensor noise before feature extraction
- Apply FFT-based frequency analysis to improve walking vs. shaking discrimination
- Use a Savitzky-Golay smoother for cleaner magnitude curves

**Event Detection**
- Replace rule-based logic with a lightweight 1D CNN or LSTM for sequence classification
- Add a hysteresis buffer to prevent rapid label switching at segment boundaries
- Introduce a `confidence_score` field based on distance from thresholds

**LlamaIndex / LLM**
- Use `llama-index` with a local model (Ollama + Llama 3) for fully offline operation
- Build a multi-turn chat interface using `CondenseQuestionChatEngine`
- Add document metadata (event type, timestamp range) for filtered queries

**Real Data**
- Ingest live BNO055 / BMI088 data via USB serial in Colab
- Support Bosch SensorTec `.csv` export format directly
- Add a streaming mode that detects events in near-real-time

**Deployment**
- Wrap the pipeline in a FastAPI microservice
- Export the rule-based classifier as a standalone Python module
- Add unit tests for each detection rule